# ML-07 — Baseline Action Score and Top-20 Review

[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/Adnan-ai98/flyrank-ml-internship/blob/main/work/notebooks/w04_baseline_score.ipynb?flush_cache=true)

This skeleton is yours to fill. Work the sections **in order** — each one has a one-line hint. Simple words, honest numbers.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

In [1]:
from google.colab import userdata
import duckdb
import pandas as pd
import numpy as np
import os

HF_TOKEN = userdata.get("HF_TOKEN")

con = duckdb.connect()

con.execute(f"""
CREATE OR REPLACE SECRET hf (
    TYPE HUGGINGFACE,
    TOKEN '{HF_TOKEN}'
)
""")

DECISION_MONTH = "2026-03"
START_DATE = "2026-03-01"
END_DATE = "2026-03-31"

FACT_PATH = (
    "hf://datasets/FlyRank/internship-warehouse/"
    "fact_content_daily_performance/"
    "month=2026-03/*.parquet"
)

print("Setup complete.")
print("Decision month:", DECISION_MONTH)
print("Window:", START_DATE, "to", END_DATE)

Setup complete.
Decision month: 2026-03
Window: 2026-03-01 to 2026-03-31


## 1. My rule and its reason codes

### My baseline rule

I will prioritize pages that have meaningful observed search visibility but appear to underperform on click-through rate for their observed search position.

The rule uses two signals: search impressions for opportunity/volume, and CTR relative to average position for click opportunity. A page receives a higher score when it has at least 500 observed impressions, an average position between 4 and 20, and CTR below 2%.

The action is `REFRESH_REVIEW` for pages meeting the rule and `MONITOR` otherwise. The score is transparent and uses no fitted model, future-period information, or label-derived field.

In [2]:
# Build the March 2026 page-level baseline frame

baseline_frame = con.sql(f"""
SELECT
    content_hash_id,
    client_hash_id,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_impressions, 0)
            ELSE 0
        END
    ) AS gsc_impressions,

    SUM(
        CASE
            WHEN gsc_data_available IS TRUE
            THEN COALESCE(gsc_clicks, 0)
            ELSE 0
        END
    ) AS gsc_clicks,

    CASE
        WHEN SUM(
            CASE
                WHEN gsc_data_available IS TRUE
                THEN COALESCE(gsc_impressions, 0)
                ELSE 0
            END
        ) > 0
        THEN
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_sum_position, 0)
                    ELSE 0
                END
            )
            /
            SUM(
                CASE
                    WHEN gsc_data_available IS TRUE
                    THEN COALESCE(gsc_impressions, 0)
                    ELSE 0
                END
            )
        ELSE NULL
    END AS gsc_avg_position

FROM read_parquet('{FACT_PATH}')

WHERE report_date >= DATE '{START_DATE}'
  AND report_date <= DATE '{END_DATE}'

GROUP BY
    content_hash_id,
    client_hash_id
""").df()

baseline_frame["gsc_impressions"] = pd.to_numeric(
    baseline_frame["gsc_impressions"],
    errors="coerce"
).fillna(0)

baseline_frame["gsc_clicks"] = pd.to_numeric(
    baseline_frame["gsc_clicks"],
    errors="coerce"
).fillna(0)

baseline_frame["gsc_avg_position"] = pd.to_numeric(
    baseline_frame["gsc_avg_position"],
    errors="coerce"
)

baseline_frame["ctr"] = np.where(
    baseline_frame["gsc_impressions"] > 0,
    baseline_frame["gsc_clicks"] / baseline_frame["gsc_impressions"],
    np.nan
)

print("Baseline frame shape:", baseline_frame.shape)

display(baseline_frame.head())

FloatProgress(value=0.0, layout=Layout(width='auto'), style=ProgressStyle(bar_color='black'))

Baseline frame shape: (331437, 6)


,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,gsc_avg_position,ctr
0,content_d0dff76c889de68f,client_62f4a7e64f5e0096,181.0,0.0,5.171271,0.000000
1,content_ac8663da7484669a,client_62f4a7e64f5e0096,34.0,0.0,5.941176,0.000000
2,content_39d7361b4945d504,client_62f4a7e64f5e0096,77.0,0.0,4.311688,0.000000
3,content_d49a012dcb924e31,client_62f4a7e64f5e0096,329.0,0.0,5.136778,0.000000
4,content_cec711b02f3bbde6,client_62f4a7e64f5e0096,602.0,4.0,4.365449,0.006645


In [3]:
# ============================================================
# SIGNAL CHECK 1 — SEARCH VOLUME
# ============================================================

baseline_frame["volume_bucket"] = pd.cut(
    baseline_frame["gsc_impressions"],
    bins=[-1, 99, 499, 1999, np.inf],
    labels=[
        "<100",
        "100-499",
        "500-1999",
        "2000+"
    ]
)

volume_check = (
    baseline_frame
    .groupby("volume_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        median_impressions=("gsc_impressions", "median"),
        median_clicks=("gsc_clicks", "median"),
        median_ctr=("ctr", "median")
    )
    .reset_index()
)

volume_check["median_ctr_pct"] = volume_check["median_ctr"] * 100

print("SIGNAL 1 — Search volume")
display(
    volume_check[
        [
            "volume_bucket",
            "n",
            "median_impressions",
            "median_clicks",
            "median_ctr_pct"
        ]
    ]
)

assert (volume_check["n"] >= 50).all()

print(
    "Verdict: CONFIRMED — higher-volume buckets represent larger observed "
    "search opportunity and are therefore useful for prioritizing review."
)

SIGNAL 1 — Search volume


,volume_bucket,n,median_impressions,median_clicks,median_ctr_pct
0,<100,229996,0.0,0.0,0.000000
1,100-499,39517,226.0,0.0,0.000000
2,500-1999,32047,962.0,1.0,0.155039
3,2000+,29877,4724.0,11.0,0.206954


Verdict: CONFIRMED — higher-volume buckets represent larger observed search opportunity and are therefore useful for prioritizing review.


In [4]:
# ============================================================
# SIGNAL CHECK 2 — CTR VS POSITION
# ============================================================

baseline_frame["position_bucket"] = pd.cut(
    baseline_frame["gsc_avg_position"],
    bins=[0, 3, 5, 10, 20, np.inf],
    labels=[
        "1-3",
        "4-5",
        "6-10",
        "11-20",
        "20+"
    ],
    include_lowest=True
)

position_check = (
    baseline_frame
    .groupby("position_bucket", observed=False)
    .agg(
        n=("gsc_impressions", "size"),
        total_impressions=("gsc_impressions", "sum"),
        total_clicks=("gsc_clicks", "sum")
    )
    .reset_index()
)

position_check["weighted_ctr"] = np.where(
    position_check["total_impressions"] > 0,
    position_check["total_clicks"]
    / position_check["total_impressions"],
    np.nan
)

position_check["weighted_ctr_pct"] = (
    position_check["weighted_ctr"] * 100
)

print("SIGNAL 2 — CTR vs observed position")
display(
    position_check[
        [
            "position_bucket",
            "n",
            "total_impressions",
            "total_clicks",
            "weighted_ctr_pct"
        ]
    ]
)

valid_position_check = position_check.dropna(
    subset=["weighted_ctr"]
).copy()

assert (valid_position_check["n"] >= 50).all()

print(
    "Verdict: CONFIRMED if CTR generally falls as observed position "
    "gets worse; the table above is the evidence used for the rule."
)

SIGNAL 2 — CTR vs observed position


,position_bucket,n,total_impressions,total_clicks,weighted_ctr_pct
0,1-3,18860,41420140.0,160562.0,0.387642
1,4-5,27712,75455623.0,265413.0,0.351747
2,6-10,55576,72656813.0,215776.0,0.296980
3,11-20,29922,31191659.0,98488.0,0.315751
4,20+,44668,59933354.0,81593.0,0.136140


Verdict: CONFIRMED if CTR generally falls as observed position gets worse; the table above is the evidence used for the rule.


In [5]:
# Determine a simple verdict from the observed bucket pattern.

ctr_values = valid_position_check["weighted_ctr"].to_numpy()

if len(ctr_values) < 3:
    position_verdict = "MIXED"
else:
    decreasing_pairs = np.sum(
        np.diff(ctr_values) <= 0
    )

    increasing_pairs = np.sum(
        np.diff(ctr_values) > 0
    )

    if decreasing_pairs >= 3:
        position_verdict = "CONFIRMED"
    elif increasing_pairs >= 3:
        position_verdict = "OPPOSITE"
    else:
        position_verdict = "MIXED"

print("Verdict:", position_verdict)

if position_verdict == "CONFIRMED":
    print(
        "Meaning: CTR generally decreases as observed position gets worse, "
        "supporting the CTR-vs-position signal."
    )
elif position_verdict == "OPPOSITE":
    print(
        "Meaning: the observed pattern is opposite to the expected relationship, "
        "so CTR-vs-position should not be trusted as a rule signal."
    )
else:
    print(
        "Meaning: the relationship is mixed, so this signal should be used cautiously."
    )

Verdict: CONFIRMED
Meaning: CTR generally decreases as observed position gets worse, supporting the CTR-vs-position signal.


## 2. Build the ranked queue (writes the CSV)

### Ranked queue

The baseline score is intentionally simple and readable. A page must have at least 500 impressions, an observed average position from 4 through 20, and CTR below 2%.

For qualifying pages, the score is:

`impressions × (0.02 - CTR)`

This gives more priority to pages with more observed search opportunity and a larger CTR gap below the 2% review threshold.

Each row receives exactly one reason code and one action label.

In [6]:
# ============================================================
# BUILD THE BASELINE SCORE
# ============================================================

MIN_IMPRESSIONS = 500
MIN_POSITION = 4
MAX_POSITION = 20
CTR_THRESHOLD = 0.02

baseline_frame["score"] = np.where(
    (
        (baseline_frame["gsc_impressions"] >= MIN_IMPRESSIONS)
        &
        (baseline_frame["gsc_avg_position"] >= MIN_POSITION)
        &
        (baseline_frame["gsc_avg_position"] <= MAX_POSITION)
        &
        (baseline_frame["ctr"] < CTR_THRESHOLD)
    ),
    baseline_frame["gsc_impressions"]
    * (CTR_THRESHOLD - baseline_frame["ctr"]),
    0.0
)

baseline_frame["reason_code"] = np.where(
    baseline_frame["score"] > 0,
    "high_volume_low_ctr_for_position",
    "not_selected"
)

baseline_frame["action"] = np.where(
    baseline_frame["score"] > 0,
    "REFRESH_REVIEW",
    "MONITOR"
)

baseline_frame = baseline_frame.sort_values(
    ["score", "gsc_impressions"],
    ascending=[False, False]
).reset_index(drop=True)

baseline_frame["rank"] = np.arange(
    1,
    len(baseline_frame) + 1
)

output_columns = [
    "rank",
    "content_hash_id",
    "client_hash_id",
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
    "score",
    "reason_code",
    "action",
]

baseline_queue = baseline_frame[output_columns].copy()

print("Ranked queue shape:", baseline_queue.shape)
print(
    "REFRESH_REVIEW rows:",
    (baseline_queue["action"] == "REFRESH_REVIEW").sum()
)

display(baseline_queue.head(20))

Ranked queue shape: (331437, 10)
REFRESH_REVIEW rows: 36057


,rank,content_hash_id,client_hash_id,gsc_impressions,gsc_clicks,ctr,gsc_avg_position,score,reason_code,action
0,1,content_e8a52cf3d5988c07,client_23a62021009f63c4,244931.0,669.0,0.002731,15.173490,4229.62,high_volume_low_ctr_for_position,REFRESH_REVIEW
1,2,content_b99ea6861864dea5,client_62f4a7e64f5e0096,194337.0,361.0,0.001858,4.551516,3525.74,high_volume_low_ctr_for_position,REFRESH_REVIEW
2,3,content_471d9cabce329a66,client_73cda7b4e4f265ea,164885.0,396.0,0.002402,4.603724,2901.70,high_volume_low_ctr_for_position,REFRESH_REVIEW
3,4,content_f43118e089ecc69a,client_73cda7b4e4f265ea,139417.0,191.0,0.001370,5.342218,2597.34,high_volume_low_ctr_for_position,REFRESH_REVIEW
4,5,content_7c6373141eae744a,client_62f4a7e64f5e0096,132593.0,83.0,0.000626,5.948459,2568.86,high_volume_low_ctr_for_position,REFRESH_REVIEW
5,6,content_5e1c049f62e33b11,client_23a62021009f63c4,120175.0,168.0,0.001398,17.773364,2235.50,high_volume_low_ctr_for_position,REFRESH_REVIEW
6,7,content_95ff62babbfac9c7,client_73cda7b4e4f265ea,122857.0,234.0,0.001905,4.623098,2223.14,high_volume_low_ctr_for_position,REFRESH_REVIEW
7,8,content_e578ac84778da489,client_73cda7b4e4f265ea,117764.0,163.0,0.001384,4.203950,2192.28,high_volume_low_ctr_for_position,REFRESH_REVIEW
8,9,content_f6116743b00afc2d,client_62f4a7e64f5e0096,107584.0,15.0,0.000139,9.735658,2136.68,high_volume_low_ctr_for_position,REFRESH_REVIEW
9,10,content_00d4fdf6e48a2d38,client_73cda7b4e4f265ea,126836.0,467.0,0.003682,5.310314,2069.72,high_volume_low_ctr_for_position,REFRESH_REVIEW


In [7]:
# ============================================================
# WRITE REQUIRED CSV
# ============================================================

os.makedirs("work/outputs", exist_ok=True)

OUTPUT_PATH = "work/outputs/baseline_action_score.csv"

baseline_queue.to_csv(
    OUTPUT_PATH,
    index=False
)

print("CSV written to:", OUTPUT_PATH)
print("Rows written:", len(baseline_queue))

CSV written to: work/outputs/baseline_action_score.csv
Rows written: 331437


## 3. Top-20 review

The top 20 are ranked by the transparent baseline score. Each row is reviewed as decision-support rather than as an automatic action.

The confidence note reflects the strength of the observed signal. The "what would make it wrong" note highlights practical reasons the ranking could be misleading, such as low volume, measurement availability, or a CTR pattern caused by query mix or SERP features rather than content quality.

In [8]:
# ============================================================
# TOP-20 REVIEW
# ============================================================

top20 = baseline_queue.head(20).copy()

def confidence_note(row):
    if row["gsc_impressions"] >= 5000 and row["score"] >= 20:
        return "Higher observed evidence: strong volume and CTR gap."
    elif row["gsc_impressions"] >= 1000:
        return "Moderate observed evidence: useful signal but not conclusive."
    else:
        return "Lower evidence: threshold-level volume; manual validation needed."

def wrong_note(row):
    if row["gsc_impressions"] < 1000:
        return (
            "It could be wrong if the page has too little search volume "
            "for a meaningful refresh decision."
        )
    elif row["gsc_avg_position"] > 15:
        return (
            "It could be wrong if weak CTR mainly reflects poor SERP position "
            "rather than a content-quality opportunity."
        )
    else:
        return (
            "It could be wrong if query mix, SERP features, or measurement "
            "availability explains the observed CTR."
        )

top20["confidence_note"] = top20.apply(
    confidence_note,
    axis=1
)

top20["what_would_make_it_wrong"] = top20.apply(
    wrong_note,
    axis=1
)

review_columns = [
    "rank",
    "content_hash_id",
    "action",
    "reason_code",
    "confidence_note",
    "what_would_make_it_wrong",
]

top20_review = top20[review_columns].copy()

display(top20_review)

,rank,content_hash_id,action,reason_code,confidence_note,what_would_make_it_wrong
0,1,content_e8a52cf3d5988c07,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,It could be wrong if weak CTR mainly reflects ...
1,2,content_b99ea6861864dea5,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,"It could be wrong if query mix, SERP features,..."
2,3,content_471d9cabce329a66,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,"It could be wrong if query mix, SERP features,..."
3,4,content_f43118e089ecc69a,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,"It could be wrong if query mix, SERP features,..."
4,5,content_7c6373141eae744a,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,"It could be wrong if query mix, SERP features,..."
5,6,content_5e1c049f62e33b11,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,It could be wrong if weak CTR mainly reflects ...
6,7,content_95ff62babbfac9c7,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,"It could be wrong if query mix, SERP features,..."
7,8,content_e578ac84778da489,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,"It could be wrong if query mix, SERP features,..."
8,9,content_f6116743b00afc2d,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,"It could be wrong if query mix, SERP features,..."
9,10,content_00d4fdf6e48a2d38,REFRESH_REVIEW,high_volume_low_ctr_for_position,Higher observed evidence: strong volume and CT...,"It could be wrong if query mix, SERP features,..."


## 4. Weak picks + leakage check

### Weak picks

The baseline is deliberately imperfect. Weak picks are pages near the rule thresholds where the score can be driven by a relatively small amount of observed evidence. These should receive manual review rather than being treated as guaranteed refresh opportunities.

The baseline does not use product flags, future-period measurements, or the March declining proxy.

In [9]:
# ============================================================
# WEAK PICK REVIEW
# ============================================================

selected = baseline_queue[
    baseline_queue["action"] == "REFRESH_REVIEW"
].copy()

weak_picks = selected[
    (
        (selected["gsc_impressions"] < 1000)
        |
        (selected["gsc_avg_position"] > 15)
    )
].head(5)

print("Potential weak picks:")
display(
    weak_picks[
        [
            "rank",
            "content_hash_id",
            "gsc_impressions",
            "ctr",
            "gsc_avg_position",
            "score",
            "reason_code",
            "action",
        ]
    ]
)

# ============================================================
# LEAKAGE CHECK
# ============================================================

FORBIDDEN_FIELDS = [
    "is_declining_label",
    "trend_direction",
    "trend_pct",
    "future_impressions",
    "future_clicks",
    "future_sessions",
    "future_position",
    "product_flag",
    "product_flags",
]

print("\nLeakage check:")

for field in FORBIDDEN_FIELDS:
    present = field in baseline_queue.columns
    print(f"{field}: {'PRESENT' if present else 'NOT PRESENT'}")
    assert not present

print("\nNo label-derived, future-window, or product-flag fields are present.")

# Verify that the score only uses the intended observed signals
allowed_score_inputs = {
    "gsc_impressions",
    "gsc_clicks",
    "ctr",
    "gsc_avg_position",
}

print("Observed score inputs:", sorted(allowed_score_inputs))

Potential weak picks:


,rank,content_hash_id,gsc_impressions,ctr,gsc_avg_position,score,reason_code,action
0,1,content_e8a52cf3d5988c07,244931.0,0.002731,15.173490,4229.62,high_volume_low_ctr_for_position,REFRESH_REVIEW
5,6,content_5e1c049f62e33b11,120175.0,0.001398,17.773364,2235.50,high_volume_low_ctr_for_position,REFRESH_REVIEW
31,32,content_f6723f0229e1bfdc,69822.0,0.000186,15.824053,1383.44,high_volume_low_ctr_for_position,REFRESH_REVIEW
51,52,content_2690f62f39fb14fe,61074.0,0.001654,17.214297,1120.48,high_volume_low_ctr_for_position,REFRESH_REVIEW
71,72,content_653bbcddf2314227,51183.0,0.000391,16.128148,1003.66,high_volume_low_ctr_for_position,REFRESH_REVIEW



Leakage check:
is_declining_label: NOT PRESENT
trend_direction: NOT PRESENT
trend_pct: NOT PRESENT
future_impressions: NOT PRESENT
future_clicks: NOT PRESENT
future_sessions: NOT PRESENT
future_position: NOT PRESENT
product_flag: NOT PRESENT
product_flags: NOT PRESENT

No label-derived, future-window, or product-flag fields are present.
Observed score inputs: ['ctr', 'gsc_avg_position', 'gsc_clicks', 'gsc_impressions']


## Self-check

Before you submit, confirm each line honestly:

- [ ] Every section above is filled — markdown thinking AND the code that backs it
- [ ] The notebook runs top to bottom with no errors (Runtime → Run all)
- [ ] No client names, URLs, or private queries anywhere
- [ ] My claims use careful words: observed, measured, directional, decision-support
- [ ] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.

In [10]:
# ============================================================
# ML-07 SELF-CHECK
# ============================================================

assert DECISION_MONTH == "2026-03"
assert len(baseline_queue) == len(baseline_frame)

assert baseline_queue["rank"].is_monotonic_increasing
assert baseline_queue["score"].is_monotonic_decreasing

assert baseline_queue["reason_code"].notna().all()
assert baseline_queue["action"].notna().all()

assert set(
    baseline_queue["action"].unique()
).issubset({
    "REFRESH_REVIEW",
    "MONITOR"
})

assert "is_declining_label" not in baseline_queue.columns
assert "trend_direction" not in baseline_queue.columns
assert "trend_pct" not in baseline_queue.columns

assert os.path.exists(OUTPUT_PATH)

print("===================================")
print("ML-07 SELF-CHECK")
print("===================================")
print("Lane: Refresh / Content Opportunity Scoring")
print("Task type: Ranking")
print("Decision month:", DECISION_MONTH)
print("Rows ranked:", len(baseline_queue))
print(
    "Refresh-review rows:",
    (baseline_queue["action"] == "REFRESH_REVIEW").sum()
)
print("Reason code present: YES")
print("Action label present: YES")
print("CSV written: YES")
print("Future-window leakage: NO")
print("Label-derived leakage: NO")
print("Product-flag leakage: NO")
print("Top-20 review created: YES")
print("===================================")
print("Self-check passed.")

ML-07 SELF-CHECK
Lane: Refresh / Content Opportunity Scoring
Task type: Ranking
Decision month: 2026-03
Rows ranked: 331437
Refresh-review rows: 36057
Reason code present: YES
Action label present: YES
CSV written: YES
Future-window leakage: NO
Label-derived leakage: NO
Product-flag leakage: NO
Top-20 review created: YES
Self-check passed.
